In [1]:
!pip install xmltodict

In [23]:
import requests
import xmltodict
import pprint
import pandas as pd
import math
import time

# 1. API 설정
# 공공데이터포털에서 '한국환경공단_전기자동차 충전소 정보' 활용 신청 후 발급받은 디코딩 키 입력
API_KEY = "본인의  API 키값 입력" 
url = "http://apis.data.go.kr/B552584/EvCharger/getChargerInfo"

params = {
    "serviceKey":API_KEY,
    "pageNo":"1",
    "numOfRows":"10", 
    "zcode":"11"
}

# 2. API 호출
response = requests.get(url, params=params)
# print(response.content)

# 3. XML 데이터를 딕셔너리로 변환
# response.text(문자열) 대신 response.content(바이트)를 사용하는 것이 인코딩 에러 방지에 좋음
dict_data = xmltodict.parse(response.content)
# print(dict_data)
# pprint.pprint(dict_data)
# print( dict_data['response']['header']['totalCount'] )

# 응답 데이터에서 전체 개수 추출 (문자열을 정수로 변환)
total_count = int(dict_data['response']['header']['totalCount'])
print(f'서울 지역 총 충전기 개수 : {total_count}')

# --- STEP 2: 총 호출해야 할 페이지 수 계산 ---
num_of_rows = 5000
total_pages = math.ceil( total_count / num_of_rows )
print(f'총 호출할 페이지 수 : {total_pages}페이지 (1페이지 당 {num_of_rows}개씩)')


# --- STEP 3: 반복문을 돌며 전체 데이터 수집 ---
all_items = []

for page in range(1, total_pages + 1):
    print(f'[{page}/{total_pages}] 페이지 수집 중...')

    params = {
        "serviceKey":API_KEY,
        "pageNo":str(page),
        "numOfRows":str(num_of_rows), 
        "zcode":"11"
    }

    try:
        res = requests.get(url, params=params)
        data = xmltodict.parse(res.content)
        items = data['response']['body']['items']['item']

        if isinstance(items, dict):
            all_items.append(items)
        else:
            all_items.extend(items)


    except Exception as e:
        print(f'오류 발생 ({page}페이지): {e}')

    time.sleep(1)
print('모든 데이터 수집 완료!!')

df_all = pd.DataFrame(all_items)

print(f'최종 데이터 크기 : {df_all.shape}')

df_all.to_csv('seoul_data.csv', index=False, encoding='utf-8-sig')
print('seoul_data.csv 파일로 저장됨.')

서울 지역 총 충전기 개수 : 71785
총 호출할 페이지 수 : 15페이지 (1페이지 당 5000개씩)
[1/15] 페이지 수집 중...
[2/15] 페이지 수집 중...
[3/15] 페이지 수집 중...
[4/15] 페이지 수집 중...
[5/15] 페이지 수집 중...
[6/15] 페이지 수집 중...
[7/15] 페이지 수집 중...
[8/15] 페이지 수집 중...
[9/15] 페이지 수집 중...
[10/15] 페이지 수집 중...
[11/15] 페이지 수집 중...
[12/15] 페이지 수집 중...
[13/15] 페이지 수집 중...
[14/15] 페이지 수집 중...
[15/15] 페이지 수집 중...
모든 데이터 수집 완료!!
최종 데이터 크기 : (71785, 37)
seoul_data.csv 파일로 저장됨.


In [24]:
df_all = pd.read_csv('seoul_data.csv', encoding='utf-8-sig', low_memory=False)
df_all.shape

(71785, 37)